In [ ]:
# %pip install pandas
# %pip install polars
# %pip install duckdb
# %pip install pyarrow
# %pip install psutil

In [ ]:
import os
import threading
import time
import duckdb
import psutil
import pandas as pd
import polars as pl

In [ ]:
pandas_benchamark = True
polars_benchamrk = False
duckdb_benchamrk = False

In [ ]:
files = [
    "iowa_sales.snappy.parquet",
    "iowa_category.snappy.parquet",
    "iowa_date.snappy.parquet",
    "iowa_store.snappy.parquet",
    "iowa_vendor.snappy.parquet",
    "iowa_item.snappy.parquet",
]

summary = []

for file in files:
    df = pd.read_parquet(file)

    summary.append(
        {
            "file": file,
            "rows": len(df),
            "columns": len(df.columns),
            "size_mb": round(os.path.getsize(file) / 1024 / 1024, 2),
        }
    )

pd.DataFrame(summary)

In [ ]:
def run_benchmark(func, name):
    process = psutil.Process(os.getpid())

    max_ram = 0
    max_cpu = 0
    running = True

    def monitor():
        nonlocal max_ram, max_cpu, running

        process.cpu_percent()  # warmup

        while running:
            ram = process.memory_info().rss / 1024**3
            cpu = process.cpu_percent()
            max_ram = max(max_ram, ram)
            max_cpu = max(max_cpu, cpu)
            time.sleep(0.1)

    monitor_thread = threading.Thread(target=monitor)
    monitor_thread.start()
    start = time.perf_counter()
    result = func()
    elapsed = time.perf_counter() - start
    running = False
    monitor_thread.join()

    return {
        "engine": name,
        "time": round(elapsed, 2),
        "peak_ram_gb": round(max_ram, 2),
        "peak_cpu_pct": round(max_cpu, 0),
        "rows": len(result),
    }

In [ ]:
def save_results(results, filename="benchmark_results.csv"):
    df_results = pd.DataFrame(results)

    avg_row = {
        "engine": df_results["engine"].iloc[0] + "_AVG",
        "time": round(df_results["time"].mean(), 2),
        "peak_ram_gb": round(df_results["peak_ram_gb"].mean(), 2),
        "peak_cpu_pct": round(df_results["peak_cpu_pct"].mean(), 0),
        "rows": int(df_results["rows"].mean())
    }

    df_results = pd.concat(
        [df_results, pd.DataFrame([avg_row])],
        ignore_index=True
    )

    df_results.to_csv(
        filename,
        mode="a",
        header=not os.path.exists(filename),
        index=False
    )

    display(df_results)

In [ ]:
def pandas_run():

    sales = pd.read_parquet("iowa_sales.snappy.parquet").drop(
        columns=["Inserted"], errors="ignore"
    )
    category = pd.read_parquet("iowa_category.snappy.parquet").drop(
        columns=["Inserted"], errors="ignore"
    )
    date = pd.read_parquet("iowa_date.snappy.parquet").drop(
        columns=["Inserted"], errors="ignore"
    )
    store = pd.read_parquet("iowa_store.snappy.parquet").drop(
        columns=["Inserted"], errors="ignore"
    )
    vendor = pd.read_parquet("iowa_vendor.snappy.parquet").drop(
        columns=["Inserted"], errors="ignore"
    )
    item = pd.read_parquet("iowa_item.snappy.parquet").drop(
        columns=["Inserted"], errors="ignore"
    )

    result = (
        sales.merge(category, on="CategoryID")
        .merge(date, on="DateID")
        .merge(store, on="StoreID")
        .merge(vendor, on="VendorID")
        .merge(item, on="ItemID")
        .query("CalendarYear >= 2022")
        .groupby(
            [
                "CalendarYear",
                "County",
                "StoreName",
                "VendorName",
                "CategoryName",
                "ItemName",
            ],
            as_index=False,
            dropna=False,
        )
        .agg(
            total_sales=("SaleDollars", "sum"),
            bottles_sold=("BottlesSold", "sum"),
            volume_liters=("VolumeSoldLiters", "sum"),
            avg_retail=("StateBottleRetail", "mean"),
            avg_cost=("StateBottleCost", "mean"),
            transactions=("ID", "count"),
        )
        .sort_values(
            [
                "CalendarYear",
                "County",
                "StoreName",
                "VendorName",
                "CategoryName",
                "ItemName",
                "total_sales",
                "bottles_sold",
            ],
            ascending=[True, True, True, True, True, True, False, False],
        )
    )

    return result

In [ ]:
def polars_run():

    sales = pl.scan_parquet("iowa_sales.snappy.parquet").drop("Inserted", strict=False)
    category = pl.scan_parquet("iowa_category.snappy.parquet").drop(
        "Inserted", strict=False
    )
    date = pl.scan_parquet("iowa_date.snappy.parquet").drop("Inserted", strict=False)
    store = pl.scan_parquet("iowa_store.snappy.parquet").drop("Inserted", strict=False)
    vendor = pl.scan_parquet("iowa_vendor.snappy.parquet").drop(
        "Inserted", strict=False
    )
    item = pl.scan_parquet("iowa_item.snappy.parquet").drop("Inserted", strict=False)

    result = (
        sales.join(category, on="CategoryID")
        .join(date, on="DateID")
        .join(store, on="StoreID")
        .join(vendor, on="VendorID")
        .join(item, on="ItemID")
        .filter(pl.col("CalendarYear") >= 2022)
        .group_by(
            [
                "CalendarYear",
                "County",
                "StoreName",
                "VendorName",
                "CategoryName",
                "ItemName",
            ]
        )
        .agg(
            [
                pl.col("SaleDollars").sum().alias("total_sales"),
                pl.col("BottlesSold").sum().alias("bottles_sold"),
                pl.col("VolumeSoldLiters").sum().alias("volume_liters"),
                pl.col("StateBottleRetail").mean().alias("avg_retail"),
                pl.col("StateBottleCost").mean().alias("avg_cost"),
                pl.col("ID").count().alias("transactions"),
            ]
        )
        .sort(
            [
                "CalendarYear",
                "County",
                "StoreName",
                "VendorName",
                "CategoryName",
                "ItemName",
                "total_sales",
                "bottles_sold",
            ],
            descending=[False, False, False, False, False, False, True, True],
        )
        .collect()
    )

    result.head(20)
    return result

In [ ]:
def duckdb_run():

    result = duckdb.sql("""
    SELECT
        d.CalendarYear,
        s.County,
        s.StoreName,
        v.VendorName,
        c.CategoryName,
        i.ItemName,
        SUM(f.SaleDollars)       AS total_sales,
        SUM(f.BottlesSold)       AS bottles_sold,
        SUM(f.VolumeSoldLiters)  AS volume_liters,
        AVG(f.StateBottleRetail) AS avg_retail,
        AVG(f.StateBottleCost)   AS avg_cost,
        COUNT(f.ID)              AS transactions
    FROM read_parquet('iowa_sales.snappy.parquet') f
    JOIN read_parquet('iowa_category.snappy.parquet') c
        ON f.CategoryID = c.CategoryID
    JOIN read_parquet('iowa_date.snappy.parquet') d
        ON f.DateID = d.DateID
    JOIN read_parquet('iowa_store.snappy.parquet') s
        ON f.StoreID = s.StoreID
    JOIN read_parquet('iowa_vendor.snappy.parquet') v
        ON f.VendorID = v.VendorID
    JOIN read_parquet('iowa_item.snappy.parquet') i
        ON f.ItemID = i.ItemID
    WHERE d.CalendarYear >= 2022
    GROUP BY
        d.CalendarYear,
        s.County,
        s.StoreName,
        v.VendorName,
        c.CategoryName,
        i.ItemName
    ORDER BY
        d.CalendarYear,
        s.County,
        s.StoreName,
        v.VendorName,
        c.CategoryName,
        i.ItemName,
        total_sales DESC,
        bottles_sold DESC
    """).df()

    return result

In [ ]:
results = []
if pandas_benchamark:
    for _ in range(5):
        results.append(run_benchmark(pandas_run, "Pandas"))
    save_results(results)

results = []
if polars_benchamrk:
    for _ in range(5):
        results.append(run_benchmark(polars_run, "Polars"))
    save_results(results)

results = []
if duckdb_benchamrk:
    for _ in range(5):
        results.append(run_benchmark(duckdb_run, "DuckDB"))
    save_results(results)

In [ ]:
pd.read_csv("benchmark_results.csv")